In [ ]:
!pip -q install --force-reinstall --no-deps 'torch==2.4.1' 'torchvision==0.19.1' 'torchaudio==2.4.1' --index-url https://download.pytorch.org/whl/cu121
!pip -q install --upgrade 'transformers==4.51.3' accelerate 'bitsandbytes==0.43.3'

In [ ]:
import os, subprocess, sys

REPO = "/kaggle/working/lawforge"
if not os.path.isdir(REPO):
    subprocess.check_call(
        ["git", "clone", "--depth", "1", "https://github.com/PAMF2/lawforge.git", REPO]
    )
sys.path.insert(0, REPO)

In [ ]:
import os, torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL = "Qwen/Qwen2.5-14B-Instruct"
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
tok = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(
    MODEL, quantization_config=bnb, device_map="cuda", trust_remote_code=True
)
model.eval()
print(f"loaded mem={torch.cuda.memory_allocated() / 1e9:.2f}GB")

In [ ]:
import json
from pathlib import Path

INPUTS = Path(f"{REPO}/kaggle/llm_propose/inputs")
rows = []
for s in ["hard2_test", "hard3_test"]:
    for line in open(INPUTS / f"{s}.jsonl"):
        r = json.loads(line)
        r["_split"] = s
        rows.append(r)
print(f"rows: {len(rows)}")

In [ ]:
import time, json, re
from pathlib import Path
from solver.counterex import parse_eq, collect_vars, satisfies, violates

SYSTEM = (
    "You search for finite magma counterexamples. Given hypothesis h and goal g, "
    "find a small finite magma (order 2 or 3, i.e. a set {0,1} or {0,1,2}) such that "
    "h holds for ALL variable choices but g FAILS for at least one. The magma is "
    "specified by its Cayley table a JSON 2D array T where T[a][b] = a*b. "
    "You MUST output exactly: REASONING (one line) then TABLE: [[...],[...]] "
    "where the array has order n rows of n integers each in [0,n). "
    "If you cannot find one in your budget, output TABLE: NONE"
)


def parse_table(txt):
    m = re.search(r"TABLE:\s*(\[\[[^\]]+\][^\]]*\])", txt, re.DOTALL)
    if not m:
        return None
    try:
        t = json.loads(m.group(1))
        n = len(t)
        if not all(isinstance(row, list) and len(row) == n for row in t):
            return None
        if not all(isinstance(x, int) and 0 <= x < n for row in t for x in row):
            return None
        return t, n
    except Exception:
        return None


def is_real_ce(h, g, table, n):
    try:
        eq1 = parse_eq(h.replace("\u25c7", "*"))
        eq2 = parse_eq(g.replace("\u25c7", "*"))
        v1 = sorted(collect_vars(eq1[0]) | collect_vars(eq1[1]))
        v2 = sorted(collect_vars(eq2[0]) | collect_vars(eq2[1]))
        return satisfies(eq1, table, n, v1) and violates(eq2, table, n, v2)
    except Exception:
        return False


@torch.inference_mode()
def propose(h, g):
    user = f"h: {h}\ng: {g}\n\nFind a finite magma (order 2 or 3) refuting h => g."
    msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": user}]
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tok(text, return_tensors="pt").to(model.device)
    out = model.generate(
        **inputs,
        max_new_tokens=300,
        do_sample=False,
        pad_token_id=tok.pad_token_id,
        eos_token_id=tok.eos_token_id,
    )
    return tok.decode(out[0][inputs["input_ids"].shape[1] :], skip_special_tokens=True)


OUT = Path("/kaggle/working/llm_propose.jsonl")
t0 = time.time()
verified_false = 0
fp = 0
with OUT.open("w") as f:
    for i, r in enumerate(rows):
        raw = propose(r["hypothesis"], r["goal"])
        parsed = parse_table(raw)
        verified = False
        if parsed:
            table, n = parsed
            verified = is_real_ce(r["hypothesis"], r["goal"], table, n)
        if verified:
            if r["label"] == "false":
                verified_false += 1
            else:
                fp += 1
        f.write(
            json.dumps(
                {
                    "id": r["id"],
                    "split": r["_split"],
                    "label": r["label"],
                    "verified_false": verified,
                    "raw": raw[-400:],
                }
            )
            + "\n"
        )
        f.flush()
        if (i + 1) % 20 == 0:
            print(
                f"[{i + 1}/{len(rows)}] verified_false={verified_false} fp={fp} t={time.time() - t0:.0f}s",
                flush=True,
            )
print(f"=== FINAL ===", flush=True)
print(f"verified counterex on label=false: {verified_false}/305", flush=True)
print(f"wrong counterex on label=true (FPs): {fp}", flush=True)
print(f"total time: {time.time() - t0:.0f}s", flush=True)